# exp030_public_sel15_pf_candidate_selector train

Audit candidate selection over the exp029 public sel15 PF/Beam OOF-like artifact.

## Contents

1. Setup and configuration
2. Input artifact check
3. Candidate selector audit
4. Metrics and artifacts


## 1. Setup and configuration


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

from candidate_selector_audit import run_audit
from settings import EXPERIMENT_NAME, ExperimentPaths, get_nested, load_config

paths = ExperimentPaths()
paths.ensure_output_dirs()
config = load_config()

feature_path = Path(get_nested(config, "data.feature_path"))
if not feature_path.is_absolute():
    feature_path = paths.root / feature_path

print("Experiment:", EXPERIMENT_NAME)
print("Route:", get_nested(config, "experiment.route"))
print("Parent:", get_nested(config, "lineage.parent"))
print("Feature path:", feature_path)
print("Artifacts:", paths.artifacts_dir)
print("Candidate count:", len(get_nested(config, "audit.candidates") or []))


## 2. Input artifact check


In [ ]:
if not feature_path.exists():
    raise FileNotFoundError(f"exp029 feature artifact not found: {feature_path}")

preview = pd.read_csv(feature_path, nrows=5)
print("Preview rows:", len(preview))
print("Columns:", len(preview.columns))
display(preview[["well_id", "fold", "row_idx", "target_tvt", "pf_pred", "last_anchor_tvt", "beam_pred"]])


## 3. Candidate selector audit


In [ ]:
summary = run_audit(paths, config, feature_path)
print(json.dumps({
    "raw_clean_cv": summary["raw_clean_cv"],
    "best_same_oof_candidate": summary["best_same_oof_candidate"],
    "best_same_oof_cv": summary["best_same_oof_cv"],
    "leave_one_original_fold_out_selection_cv": summary["leave_one_original_fold_out_selection_cv"],
    "well_hash_holdout_selection_cv": summary["well_hash_holdout_selection_cv"],
    "selector_supported": summary["selector_supported"],
}, indent=2))


## 4. Metrics and artifacts


In [ ]:
metrics = pd.read_csv(paths.artifacts_dir / "candidate_selector_metrics.csv")
selection = pd.read_csv(paths.artifacts_dir / "candidate_selector_selection.csv")
display(metrics.head(10))
display(selection.head(20))
print("Metrics written:", paths.metrics_path)
print("Artifacts written:", paths.artifacts_dir)
